# LP-FT RAD-DINO — MV7:3 + **DB Loss** + **ML-Decoder head**

Nâng cấp bản MV7:3 study-level:
- **Loss = Distribution-Balanced Loss** (Wu ECCV'20): rebalancing weight (bù lệch co-occurrence) + negative-tolerant regularization (giảm đè nén nhãn hiếm). Thay Asymmetric Loss.
- **Head = ML-Decoder** (Ridnik'21): 13 class-query **cross-attention lên PATCH tokens** (localization) → mỗi bệnh 1 vector → group-FC ra logit. Thay linear trên CLS.
- **Gộp 7:3 ở mức feature decoder**: backbone→patch mỗi view → ML-Decoder chia sẻ → `0.7·F+0.3·L` → logit.
- **LP**: train ML-Decoder trên backbone ĐÓNG BĂNG (forward cache, detach). **FT**: mở last-3 block + decoder.
- Giữ: high-LR, epoch nhiều, TẮT early-stop (lưu best+last). Output file MỚI.
> Cell 1–3b (env/split/cache/study-table) **GIỐNG bản MV73** — nếu đã chạy ở notebook MV73 cùng phiên, chỉ cần chạy lại chúng ở đây để có biến.

## 1. Env + config

In [ ]:
import os, io, time, glob, math, shutil, random, numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F
from PIL import Image
import torchvision.transforms as T
from transformers import AutoModel, AutoImageProcessor
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from google.colab import drive
try: drive.mount('/content/drive')
except Exception as e: print('drive:', e)

SRC_DIR   = '/content/drive/MyDrive/MIMIC_CXR'
WRITE_DIR = '/content/drive/MyDrive/lpft_out'
CACHE_DIR = '/content/drive/MyDrive/lpft_cache256'
LOCAL_CACHE = '/content/lpft_cache256'
for d in (WRITE_DIR, CACHE_DIR, LOCAL_CACHE): os.makedirs(d, exist_ok=True)
REPO=os.environ.get('RADDINO_REPO','/content/rad-dino') if os.path.exists('/content/rad-dino') else 'microsoft/rad-dino'
BILLING_PROJECT='mimic-ehr-v310'; BUCKET='mimic-cxr-jpg-2.1.0.physionet.org'
THREADS=32; CACHE_RES=256; TRAIN_RES=224; CACHE_SHARD=4000; N_UNFREEZE=3
WF, WL = 0.7, 0.3
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device, '| REPO:', REPO)
proc=AutoImageProcessor.from_pretrained(REPO); MEAN, STD = proc.image_mean, proc.image_std
CHEXPERT_LABELS=['Atelectasis','Cardiomegaly','Consolidation','Edema','Enlarged Cardiomediastinum',
 'Fracture','Lung Lesion','Lung Opacity','No Finding','Pleural Effusion','Pleural Other','Pneumonia','Pneumothorax','Support Devices']
LABELS=[c for c in CHEXPERT_LABELS if c!='Support Devices']; C=len(LABELS)
def resolve_uones(L): return np.where(np.isnan(L),0.0,np.where(L==-1,1.0,L)).astype('float32')
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, fbeta_score
GRID=np.arange(0.05,0.96,0.01)
def bthr(y,p): s=[fbeta_score(y,(p>=t).astype(int),beta=2,zero_division=0) for t in GRID]; return float(GRID[int(np.argmax(s))])
def macro_ap_from_P(P, Lraw):
    aps=[]
    for j in range(C):
        col=Lraw[:,j]; keep=col!=-1; yt=np.where(np.isnan(col[keep]),0.,col[keep]).astype(int)
        if yt.sum()>0: aps.append(average_precision_score(yt, P[keep,j]))
    return float(np.mean(aps))
print('LABELS:', C)

## 2. cm + split subject 70/10/20 (frontal+lateral)

In [ ]:
cm = pd.read_parquet(os.path.join(SRC_DIR,'manifest_cohort.parquet'))
cm['rel'] = cm['jpg_path'].str.split(BUCKET+'/').str[-1]
rng=np.random.default_rng(42); uniq=np.unique(cm.subject_id.values); rng.shuffle(uniq)
n=len(uniq); s1,s2=int(.70*n),int(.80*n)
tr_s,va_s,te_s=set(uniq[:s1]),set(uniq[s1:s2]),set(uniq[s2:])
cm['mysplit']=cm.subject_id.map(lambda s:'train' if s in tr_s else ('validate' if s in va_s else 'test'))
print('subject 70/10/20 (ảnh):', cm.mysplit.value_counts().to_dict())

## 3. Cache 256px frontal+lateral (tái dùng cache cũ) + nạp local + bảng study

In [ ]:
# --- cache (giống MV73; nếu cache cũ đã đủ thì chỉ copy về local) ---
sel = cm[cm.view.isin(['frontal','lateral'])].reset_index(drop=True)
from google.cloud import storage
import google.auth
from google.auth.transport.requests import AuthorizedSession
from requests.adapters import HTTPAdapter
from concurrent.futures import ThreadPoolExecutor
tf256=T.Compose([T.Resize(CACHE_RES), T.CenterCrop(CACHE_RES)]); _bucket=None
def _get_bucket():
    global _bucket
    if _bucket is None:
        creds,_=google.auth.default(); sess=AuthorizedSession(creds)
        sess.mount('https://', HTTPAdapter(pool_connections=THREADS,pool_maxsize=THREADS))
        _bucket=storage.Client(project=BILLING_PROJECT,credentials=creds,_http=sess).bucket(BUCKET,user_project=BILLING_PROJECT)
    return _bucket
def _one256(rel):
    try: return np.asarray(tf256(Image.open(io.BytesIO(_get_bucket().blob(rel).download_as_bytes())).convert('RGB')),dtype=np.uint8)
    except Exception: return None
class BatchDS(Dataset):
    def __init__(s,df,bs): s.rel=df.rel.tolist(); s.id=df.dicom_id.tolist(); s.bs=bs
    def __len__(s): return (len(s.id)+s.bs-1)//s.bs
    def __getitem__(s,b):
        lo=b*s.bs; hi=min(lo+s.bs,len(s.id)); ids=s.id[lo:hi]; rels=s.rel[lo:hi]
        with ThreadPoolExecutor(THREADS) as ex: ims=list(ex.map(_one256,rels))
        keep=[(i,im) for i,im in zip(ids,ims) if im is not None]
        return ([k[0] for k in keep], np.stack([k[1] for k in keep])) if keep else ([],None)
done=set()
for f in glob.glob(os.path.join(CACHE_DIR,'cacheidx_*.parquet')): done|=set(pd.read_parquet(f).dicom_id)
todo=sel[~sel.dicom_id.isin(done)].reset_index(drop=True); print('đã cache:',len(done),'| còn:',len(todo))
sh=len(glob.glob(os.path.join(CACHE_DIR,'cache_*.npy'))); buf_id=[]; buf_im=[]
def flush():
    global sh,buf_id,buf_im
    np.save(os.path.join(CACHE_DIR,f'cache_{sh:05d}.npy'), np.concatenate(buf_im,0))
    pd.DataFrame([{'dicom_id':d,'shard':sh,'row':k} for k,d in enumerate(buf_id)]).to_parquet(os.path.join(CACHE_DIR,f'cacheidx_{sh:05d}.parquet'),index=False)
    sh+=1; buf_id=[]; buf_im=[]
if len(todo):
    loader=DataLoader(BatchDS(todo,256),batch_size=None,num_workers=8,prefetch_factor=4)
    seen=0; t0=time.time(); pbar=tqdm(loader,total=len(loader))
    for ids,ims in pbar:
        if ims is None: continue
        buf_id+=list(ids); buf_im.append(ims); seen+=len(ids); pbar.set_postfix(img_s=round(seen/(time.time()-t0),1))
        if sum(a.shape[0] for a in buf_im)>=CACHE_SHARD: flush()
    if buf_id: flush()
cidx=pd.concat([pd.read_parquet(f) for f in sorted(glob.glob(os.path.join(CACHE_DIR,'cacheidx_*.parquet')))],ignore_index=True)
cidx.to_parquet(os.path.join(CACHE_DIR,'cache_index.parquet'),index=False)
for f in sorted(glob.glob(os.path.join(CACHE_DIR,'cache_*.npy'))):
    dst=os.path.join(LOCAL_CACHE,os.path.basename(f))
    if not os.path.exists(dst): shutil.copy2(f,dst)
shutil.copy2(os.path.join(CACHE_DIR,'cache_index.parquet'),os.path.join(LOCAL_CACHE,'cache_index.parquet'))
print('cache local:', len(glob.glob(os.path.join(LOCAL_CACHE,'cache_*.npy'))),'shard')

In [ ]:
# --- nạp cache local + dựng bảng study ---
cidx=pd.read_parquet(os.path.join(LOCAL_CACHE,'cache_index.parquet'))
shard_arrs={int(s): np.load(os.path.join(LOCAL_CACHE,f'cache_{int(s):05d}.npy'),mmap_mode='r') for s in cidx.shard.unique()}
id2loc={d:(int(s),int(r)) for d,s,r in zip(cidx.dicom_id,cidx.shard,cidx.row)}
inc = cm[cm.view.isin(['frontal','lateral']) & cm.dicom_id.isin(id2loc)]
study2f = inc[inc.view=='frontal'].groupby('study_id').dicom_id.apply(list).to_dict()
study2l = inc[inc.view=='lateral'].groupby('study_id').dicom_id.apply(list).to_dict()
st = cm.drop_duplicates('study_id')[['study_id','subject_id','mysplit']+LABELS].copy()
st = st[st.study_id.isin(set(study2f)|set(study2l))].reset_index(drop=True)
st['has_f']=st.study_id.isin(study2f).astype(float); st['has_l']=st.study_id.isin(study2l).astype(float)
trm=(st.mysplit=='train').values; vam=(st.mysplit=='validate').values; tem=(st.mysplit=='test').values
print('studies:',len(st),'| train/val/test:',int(trm.sum()),int(vam.sum()),int(tem.sum()),
      '| cả 2 view:',int((st.has_f.astype(bool)&st.has_l.astype(bool)).sum()))

## 4. DB Loss + ML-Decoder + backbone + StudyDS
- **DBLoss**: `rebalance_weight` (từ tần suất lớp) × NTR (bias theo lớp + neg_scale).
- **MLDecoderHead**: 13 class-query → cross-attn lên patch tokens → group-FC.

In [ ]:
# ---------- DB Loss ----------
class DBLoss(nn.Module):
    def __init__(s, class_freq, train_num, map_a=0.1, map_b=10.0, map_g=0.3,
                 neg_scale=2.0, init_bias=0.05, eps=1e-6):
        super().__init__()
        cf=torch.tensor(class_freq,dtype=torch.float32)
        s.register_buffer('freq_inv',1.0/(cf+eps))
        s.register_buffer('bias',-torch.log(train_num/(cf+eps)-1)*init_bias)   # ν_k theo prior lớp
        s.a,s.b,s.g,s.neg,s.eps=map_a,map_b,map_g,neg_scale,eps
    def rw(s,y):
        repeat=(y*s.freq_inv).sum(1,keepdim=True)              # (B,1)
        r=s.freq_inv/(repeat+s.eps)                            # (B,C)
        return s.a+torch.sigmoid(s.b*(r-s.g))                  # map -> [a, a+1]
    def forward(s,z,y):
        w=s.rw(y); zb=z-s.bias
        Lpos=y*F.logsigmoid(zb)                                # -> log σ(z-ν)
        Lneg=(1-y)*(1.0/s.neg)*F.logsigmoid(-s.neg*zb)         # (1/λ) log σ(-λ(z-ν))  (negative-tolerant)
        return -(w*(Lpos+Lneg)).mean()

# ---------- ML-Decoder head ----------
class MLDecoderHead(nn.Module):
    def __init__(s, C, dim=768, d=768, heads=8, ff=2048, p=0.1):
        super().__init__()
        s.q=nn.Parameter(torch.randn(C,d)*0.02)
        s.kv=nn.Linear(dim,d); s.attn=nn.MultiheadAttention(d,heads,dropout=p,batch_first=True)
        s.ln1=nn.LayerNorm(d); s.ln2=nn.LayerNorm(d)
        s.ff=nn.Sequential(nn.Linear(d,ff),nn.GELU(),nn.Dropout(p),nn.Linear(ff,d))
        s.proj=nn.Parameter(torch.randn(C,d)*0.02); s.b=nn.Parameter(torch.zeros(C))
    def features(s,tokens):                                    # (B,N,dim)->(B,C,d)
        B=tokens.size(0); q=s.q.unsqueeze(0).expand(B,-1,-1); kv=s.kv(tokens)
        a,_=s.attn(q,kv,kv,need_weights=False); x=s.ln1(q+a); x=s.ln2(x+s.ff(x)); return x
    def logits(s,feat): return (feat*s.proj).sum(-1)+s.b       # (B,C,d)->(B,C)

# ---------- backbone + dataset ----------
model=AutoModel.from_pretrained(REPO).to(device)
dec=MLDecoderHead(C).to(device)
NORM=T.Normalize(MEAN,STD)
aug_tr=T.Compose([T.RandomResizedCrop(TRAIN_RES,scale=(0.8,1.0),ratio=(0.9,1.1)),T.ToTensor(),NORM])
aug_ev=T.Compose([T.Resize(TRAIN_RES),T.CenterCrop(TRAIN_RES),T.ToTensor(),NORM])
ZERO=torch.zeros(3,TRAIN_RES,TRAIN_RES)
def load_img(did,tf): sd,rw=id2loc[did]; return tf(Image.fromarray(np.array(shard_arrs[sd][rw])))
class StudyDS(Dataset):
    def __init__(s,df,train):
        s.sid=df.study_id.tolist(); s.hf=df.has_f.to_numpy('float32'); s.hl=df.has_l.to_numpy('float32')
        s.y=resolve_uones(df[LABELS].to_numpy('float32')); s.tf=aug_tr if train else aug_ev; s.train=train
    def __len__(s): return len(s.sid)
    def __getitem__(s,i):
        sid=s.sid[i]; fids=study2f.get(sid,[]); lids=study2l.get(sid,[])
        if s.train: fi=random.choice(fids) if fids else None; li=random.choice(lids) if lids else None
        else:       fi=fids[0] if fids else None;             li=lids[0] if lids else None
        xf=load_img(fi,s.tf) if fi else ZERO.clone(); xl=load_img(li,s.tf) if li else ZERO.clone()
        return xf,xl,torch.tensor(s.hf[i]),torch.tensor(s.hl[i]),torch.tensor(s.y[i])

def patches(x): return model(pixel_values=x,interpolate_pos_encoding=True).last_hidden_state[:,1:]  # bỏ CLS
def fwd(xf,xl,hf,hl,bb_grad):
    if bb_grad: tf_=patches(xf); tl_=patches(xl)
    else:
        with torch.no_grad(): tf_=patches(xf); tl_=patches(xl)
    Ff=dec.features(tf_); Fl=dec.features(tl_)                 # (B,C,d) mỗi view
    hf=hf[:,None,None]; hl=hl[:,None,None]
    feat=(WF*Ff*hf+WL*Fl*hl)/(WF*hf+WL*hl+1e-8)                # gộp 7:3 mức feature decoder
    return dec.logits(feat)
def eval_P(loader,df):
    model.eval(); dec.eval(); ps=[]
    with torch.no_grad(), torch.autocast('cuda',dtype=torch.bfloat16):
        for xf,xl,hf,hl,y in loader:
            ps.append(torch.sigmoid(fwd(xf.to(device),xl.to(device),hf.to(device),hl.to(device),False)).float().cpu().numpy())
    P=np.concatenate(ps,0); return P, macro_ap_from_P(P, df[LABELS].to_numpy('float32'))

# tần suất lớp (U-Ones) trên TRAIN cho DB Loss
Ytr=resolve_uones(st[trm][LABELS].to_numpy('float32'))
class_freq=Ytr.sum(0); train_num=len(Ytr)
crit=DBLoss(class_freq, train_num).to(device)
print('class_freq (train U-Ones):', dict(zip(LABELS,class_freq.astype(int))))

## 4b. Pha 1 — LP (backbone ĐÓNG BĂNG, train ML-Decoder), DB Loss, no early-stop

In [ ]:
for p in model.parameters(): p.requires_grad=False           # backbone frozen
STUDY_BATCH=32
tr_df=st[trm].reset_index(drop=True); va_df=st[vam].reset_index(drop=True); te_df=st[tem].reset_index(drop=True)
trl=DataLoader(StudyDS(tr_df,True), batch_size=STUDY_BATCH, shuffle=True, num_workers=8, pin_memory=True, drop_last=True, persistent_workers=True)
vll=DataLoader(StudyDS(va_df,False),batch_size=STUDY_BATCH*2, num_workers=8, pin_memory=True, persistent_workers=True)
opt=torch.optim.AdamW(dec.parameters(), lr=2e-3, weight_decay=0.05)   # LR cao cho head
EPOCHS_LP=12                                                          # decoder cần vài epoch (đắt hơn linear)
best=-1; best_state=None
for ep in range(EPOCHS_LP):
    model.eval(); dec.train(); t0=time.time()
    for xf,xl,hf,hl,y in tqdm(trl,desc=f'LP ep{ep}'):
        xf=xf.to(device); xl=xl.to(device); hf=hf.to(device); hl=hl.to(device); y=y.to(device)
        opt.zero_grad()
        with torch.autocast('cuda',dtype=torch.bfloat16): loss=crit(fwd(xf,xl,hf,hl,False),y)
        loss.backward(); opt.step()
    _,ap=eval_P(vll,va_df); print(f'LP ep{ep} val macro-AUPRC={ap:.4f} ({time.time()-t0:.0f}s)')
    if ap>best: best=ap; best_state={k:v.detach().cpu().clone() for k,v in dec.state_dict().items()}
dec.load_state_dict(best_state); torch.save(dec.state_dict(), os.path.join(WRITE_DIR,'mldec_lp_mv73.pt'))
print('LP best val macro-AUPRC:', round(best,4))

## 5. Pha 2 — FT (last-3 block + ML-Decoder), DB Loss, high-LR, no early-stop

In [ ]:
for p in model.parameters(): p.requires_grad=False
for blk in model.encoder.layer[-N_UNFREEZE:]:
    for p in blk.parameters(): p.requires_grad=True
for p in model.layernorm.parameters(): p.requires_grad=True
dec.load_state_dict(torch.load(os.path.join(WRITE_DIR,'mldec_lp_mv73.pt')))
print('trainable:', sum(p.numel() for p in model.parameters() if p.requires_grad)+sum(p.numel() for p in dec.parameters()))
opt=torch.optim.AdamW([{'params':[p for p in model.parameters() if p.requires_grad],'lr':3e-5},
                       {'params':dec.parameters(),'lr':3e-4}], weight_decay=0.05)
EPOCHS_FT=20
sched=torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[3e-5,3e-4], total_steps=len(trl)*EPOCHS_FT, pct_start=0.1)
best=-1; best_ep=-1
for ep in range(EPOCHS_FT):
    model.train(); dec.train(); t0=time.time()
    for xf,xl,hf,hl,y in tqdm(trl,desc=f'FT ep{ep}'):
        xf=xf.to(device,non_blocking=True); xl=xl.to(device,non_blocking=True)
        hf=hf.to(device); hl=hl.to(device); y=y.to(device,non_blocking=True)
        opt.zero_grad()
        with torch.autocast('cuda',dtype=torch.bfloat16): loss=crit(fwd(xf,xl,hf,hl,True),y)   # backbone có grad
        loss.backward(); opt.step(); sched.step()
    _,ap=eval_P(vll,va_df); print(f'FT ep{ep} val macro-AUPRC={ap:.4f} ({time.time()-t0:.0f}s)')
    ckpt={'backbone':model.state_dict(),'mldec':dec.state_dict(),'val_auprc':ap,'epoch':ep,
          'labels':LABELS,'N_UNFREEZE':N_UNFREEZE,'WF':WF,'WL':WL}
    torch.save(ckpt, os.path.join(WRITE_DIR,'raddino_lpft_mv73_dbml_last.pt'))
    if ap>best: best=ap; best_ep=ep; torch.save(ckpt, os.path.join(WRITE_DIR,'raddino_lpft_mv73_dbml_best.pt')); print('  ✓ BEST')
print(f'FT xong. best val macro-AUPRC={best:.4f} @ ep{best_ep}')

## 6. Eval test study-level — per-label AUROC/AUPRC/precision/recall/F2

In [ ]:
ck=torch.load(os.path.join(WRITE_DIR,'raddino_lpft_mv73_dbml_best.pt'))
model.load_state_dict(ck['backbone']); dec.load_state_dict(ck['mldec'])
tel=DataLoader(StudyDS(te_df,False), batch_size=STUDY_BATCH*2, num_workers=8, pin_memory=True)
Pte,_=eval_P(tel,te_df); Pva,_=eval_P(vll,va_df)
Lr_te=te_df[LABELS].to_numpy('float32'); Lr_va=va_df[LABELS].to_numpy('float32'); rows=[]
for j,c in enumerate(LABELS):
    cv=Lr_va[:,j]; kv=cv!=-1; yv=np.where(np.isnan(cv[kv]),0.,cv[kv]).astype(int); pv=Pva[kv,j]
    ct=Lr_te[:,j]; kt=ct!=-1; yt=np.where(np.isnan(ct[kt]),0.,ct[kt]).astype(int); pt=Pte[kt,j]
    thr=bthr(yv,pv); pred=(pt>=thr).astype(int)
    rows.append(dict(label=c,
        auroc=roc_auc_score(yt,pt) if 0<yt.sum()<len(yt) else np.nan,
        auprc=average_precision_score(yt,pt) if yt.sum()>0 else np.nan,
        precision=precision_score(yt,pred,zero_division=0), recall=recall_score(yt,pred,zero_division=0),
        f2=fbeta_score(yt,pred,beta=2,zero_division=0), thr=round(thr,3),
        prevalence=round(float(yt.mean()),4), n_pos=int(yt.sum())))
mt=pd.DataFrame(rows); mt.to_csv(os.path.join(WRITE_DIR,'lpft_metrics_mv73_dbml.csv'),index=False)
print(mt.to_string(index=False))
print('\nMACRO (MV7:3 + DBLoss + MLDecoder):', mt[['auroc','auprc','precision','recall','f2']].mean().round(4).to_dict())
print('(so bản MV7:3+ASL+linear để thấy DBLoss/MLDecoder có nhích nhãn hiếm không)')